In [1]:
import os
import random
import h5py

import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
# from torch.optim.lr_scheduler import ReduceLROnPlateau
import torchvision.models as models
from torchinfo import summary
from sklearn.metrics import r2_score
import plotly.express as px


from sklearn.preprocessing import LabelEncoder

import torchvision.transforms.functional as TF

import seaborn as sns
import matplotlib.pyplot as plt


sns.set_theme(style="ticks", palette="pastel", rc={"lines.linewidth": 2.5})

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
if hasattr(torch.backends, "cudnn"):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
try:
    torch.use_deterministic_algorithms(True)
except Exception:
    pass

generator = torch.Generator().manual_seed(SEED)

# Set the device to use for training
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# Data

In [2]:
folder_path = "models/"

# Create the folder path and checkpoints directory if they don't exist
if not os.path.exists(folder_path):
    os.makedirs(folder_path)

if not os.path.exists(os.path.join(folder_path, "checkpoints")):
    os.makedirs(os.path.join(folder_path, "checkpoints"))

- Reading files

In [3]:
# modify so indices is the index of the dataframe
def read_hdf5_to_dataframe_with_index(h5_path="unified_parallel.h5"):
    with h5py.File(h5_path, "r") as f:
        viirs_start = f["viirs_start"][:]
        viirs_end = f["viirs_end"][:]
        rgb = f["rgb"][:]
        figures = f["figures"][:]
        indices = f["indices"][:]
        iso3 = f["iso3"][:]
        types = f["type"][:]

    # Decode bytes to strings for iso3
    iso3_decoded = [x.decode("utf-8") if isinstance(x, bytes) else x for x in iso3]
    types_decoded = [x.decode("utf-8") if isinstance(x, bytes) else x for x in types]

    # Create a DataFrame with indices as the index
    df = pd.DataFrame(
        {
            "viirs_start": list(viirs_start),
            "viirs_end": list(viirs_end),
            "rgb": list(rgb),
            "figures": figures,
            "iso3": iso3_decoded,
            "type": types_decoded,
        },
        index=indices,
    )

    df.sort_index(inplace=True)  # Ensure indices are sorted

    return df

In [ ]:
path = "../src/data/processed/disaster.h5"
df = read_hdf5_to_dataframe_with_index(path)

- encode the iso3

In [ ]:
def replace_iso3_with_embedding(df, embedding_dim=64):
    le = LabelEncoder()
    df["iso3_id"] = le.fit_transform(df["iso3"])
    emb = nn.Embedding(
        num_embeddings=df["iso3_id"].nunique(), embedding_dim=embedding_dim
    )
    with torch.no_grad():
        embeddings = emb(torch.tensor(df["iso3_id"].values)).numpy()
    df["iso3_encoded"] = embeddings.tolist()
    # drop iso3_id
    df.drop(columns=["iso3_id"], inplace=True)
    return df, le, emb

df, le, emb = replace_iso3_with_embedding(df)

- Process to create virrs (end - start)

In [ ]:
def create_viirs_diff_column(df):
    df["viirs_diff"] = df["viirs_end"] - df["viirs_start"]
    df.drop(columns=["viirs_start", "viirs_end"], inplace=True)
    # transpose the rgb column to have the shape (3, 224, 224) and viirs_diff to (3, 224, 224)
    df["rgb"] = df["rgb"].apply(
        lambda x: np.transpose(x, (2, 0, 1)) if len(x.shape) == 3 else x
    )
    return df

- Normalization

In [ ]:
normalization_stats = {}

def log_and_normalize_column(df, column_name):
    df[column_name] = np.log1p(df[column_name])
    mean = df[column_name].mean()
    std = df[column_name].std()
    normalization_stats[column_name] = {"mean": mean, "std": std}
    df[column_name] = (df[column_name] - mean) / std
    return df

In [ ]:
df = create_viirs_diff_column(df)
df = log_and_normalize_column(df, "figures")

In [ ]:
df

In [ ]:
# plot histogram of iso3, I want to see the distribution of iso3
def plot_iso3_distribution(df):
    iso3_counts = df["iso3"].value_counts()
    fig = px.bar(
        iso3_counts,
        x=iso3_counts.index,
        y=iso3_counts.values,
        labels={"x": "ISO3 Code", "y": "Count"},
        title="Distribution of ISO3 Codes",
    )
    fig.update_layout(xaxis_tickangle=-45)
    fig.show()
plot_iso3_distribution(df)

In [ ]:
def plot_iso3_distribution(df):
    iso3_counts = df["iso3"].value_counts().reset_index()
    iso3_counts.columns = ["iso3", "count"]
    iso3_counts["log_count"] = np.log1p(iso3_counts["count"])

    fig = px.choropleth(
        iso3_counts,
        locations="iso3",
        color="log_count",
        hover_name="iso3",
        hover_data={"count": True, "log_count": False},
        color_continuous_scale="Viridis",
        title="Distribution of ISO3 Codes (Choropleth Map)",
    )

    # Flat map + hide Antarctica by limiting latitude
    fig.update_geos(
        projection_type="equirectangular",
        lataxis_range=[-58, 90],  # cuts off Antarctica
        showframe=False,
        showcoastlines=True,
    )

    # Move colorbar (count) closer to the map
    fig.update_layout(
        coloraxis_colorbar=dict(
            title="Log Event Count",
            # x=0,      # closer to plot area
            xanchor="left",
            y=0.5,
            len=0.8,
            thickness=14,
        ),
        margin=dict(l=0, r=20, t=50, b=0),
    )

    fig.show()


plot_iso3_distribution(df)

In [ ]:
# list lest frequent iso3 codes
def list_least_frequent_iso3(df, threshold=10):
    iso3_counts = df["iso3"].value_counts()
    least_frequent_iso3 = iso3_counts[iso3_counts < threshold].index.tolist()
    return least_frequent_iso3
print("Least frequent ISO3 codes:", list_least_frequent_iso3(df, threshold=10))

# Simple baseline

In [ ]:
# using simple linear regression to predict figures from viirs_diff and iso3_encoded
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

In [ ]:
# Step 1: Baseline with ISO3 embedding only
print("="*60)
print("BASELINE 1: ISO3 Embedding Only")
print("="*60)

X_iso3_features = []
for idx in df.index:
    iso3_enc = np.array(df.loc[idx, 'iso3_encoded'])  # shape: (64,)
    X_iso3_features.append(iso3_enc)

X_iso3 = np.array(X_iso3_features)
y = df['figures'].values

print(f"Feature matrix shape: {X_iso3.shape}")
print(f"Target shape: {y.shape}")

# Train-test-validation split
total = len(df)
train_size = int(0.7 * total)
val_size = int(0.15 * total)
test_size = total - train_size - val_size

# Create indices and shuffle
indices = np.arange(total)
np.random.seed(SEED)
np.random.shuffle(indices)

train_idx = indices[:train_size]
val_idx = indices[train_size:train_size + val_size]
test_idx = indices[train_size + val_size:]

X_iso3_train, y_train = X_iso3[train_idx], y[train_idx]
X_iso3_val, y_val = X_iso3[val_idx], y[val_idx]
X_iso3_test, y_test = X_iso3[test_idx], y[test_idx]

# Train linear regression with ISO3 only
lr_iso3 = LinearRegression()
lr_iso3.fit(X_iso3_train, y_train)

y_iso3_train_pred = lr_iso3.predict(X_iso3_train)
y_iso3_val_pred = lr_iso3.predict(X_iso3_val)
y_iso3_test_pred = lr_iso3.predict(X_iso3_test)

iso3_train_r2 = r2_score(y_train, y_iso3_train_pred)
iso3_train_mae = np.mean(np.abs(y_train - y_iso3_train_pred))

iso3_val_r2 = r2_score(y_val, y_iso3_val_pred)
iso3_val_mae = np.mean(np.abs(y_val - y_iso3_val_pred))

iso3_test_r2 = r2_score(y_test, y_iso3_test_pred)
iso3_test_mae = np.mean(np.abs(y_test - y_iso3_test_pred))

print(f"\nISO3-Only Results:")
print(f"Train R²: {iso3_train_r2:.4f}, Train MAE: {iso3_train_mae:.4f}")
print(f"Val R²:   {iso3_val_r2:.4f}, Val MAE:   {iso3_val_mae:.4f}")
print(f"Test R²:  {iso3_test_r2:.4f}, Test MAE:  {iso3_test_mae:.4f}")

# Step 2: Add VIIRS features
print("\n" + "="*60)
print("BASELINE 2: ISO3 + VIIRS with Ridge Regression")
print("="*60)

X_features = []
for idx in df.index:
    iso3_enc = np.array(df.loc[idx, 'iso3_encoded'])  # shape: (64,)
    viirs_diff = np.array(df.loc[idx, 'viirs_diff'])  # shape: (3, 224, 224)
    
    # Flatten viirs_diff
    viirs_flat = viirs_diff.flatten()  # shape: (150528,)
    
    # Combine features - iso3 first, then viirs
    combined = np.concatenate([iso3_enc, viirs_flat])
    X_features.append(combined)

X_full = np.array(X_features)

X_full_train, y_train = X_full[train_idx], y[train_idx]
X_full_val, y_val = X_full[val_idx], y[val_idx]
X_full_test, y_test = X_full[test_idx], y[test_idx]

print(f"\nFeature matrix shape: {X_full.shape}")
print(f"Train: {X_full_train.shape}, Val: {X_full_val.shape}, Test: {X_full_test.shape}")

# Use Ridge regression with regularization to handle high dimensionality
scaler = StandardScaler()
X_full_train_scaled = scaler.fit_transform(X_full_train)
X_full_val_scaled = scaler.transform(X_full_val)
X_full_test_scaled = scaler.transform(X_full_test)

# Ridge regression with cross-validation
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_full_train_scaled, y_train)

y_ridge_train_pred = ridge_model.predict(X_full_train_scaled)
y_ridge_val_pred = ridge_model.predict(X_full_val_scaled)
y_ridge_test_pred = ridge_model.predict(X_full_test_scaled)

ridge_train_r2 = r2_score(y_train, y_ridge_train_pred)
ridge_train_mae = np.mean(np.abs(y_train - y_ridge_train_pred))

ridge_val_r2 = r2_score(y_val, y_ridge_val_pred)
ridge_val_mae = np.mean(np.abs(y_val - y_ridge_val_pred))

ridge_test_r2 = r2_score(y_test, y_ridge_test_pred)
ridge_test_mae = np.mean(np.abs(y_test - y_ridge_test_pred))

print(f"\nRidge Regression (ISO3 + VIIRS) Results:")
print(f"Train R²: {ridge_train_r2:.4f}, Train MAE: {ridge_train_mae:.4f}")
print(f"Val R²:   {ridge_val_r2:.4f}, Val MAE:   {ridge_val_mae:.4f}")
print(f"Test R²:  {ridge_test_r2:.4f}, Test MAE:  {ridge_test_mae:.4f}")

# Select best model
print("\n" + "="*60)
print("MODEL COMPARISON")
print("="*60)
print(f"ISO3-Only (Linear)         - Test R²: {iso3_test_r2:.4f}")
print(f"ISO3 + VIIRS (Ridge)       - Test R²: {ridge_test_r2:.4f}")

if ridge_test_r2 > iso3_test_r2:
    print("\n✓ Selected: Ridge Regression (ISO3 + VIIRS)")
    lr_model = ridge_model
    y_train_pred = y_ridge_train_pred
    y_val_pred = y_ridge_val_pred
    y_test_pred = y_ridge_test_pred
    train_r2, val_r2, test_r2 = ridge_train_r2, ridge_val_r2, ridge_test_r2
    train_mae, val_mae, test_mae = ridge_train_mae, ridge_val_mae, ridge_test_mae
else:
    print("\n✓ Selected: Linear Regression (ISO3 Only)")
    lr_model = lr_iso3
    y_train_pred = y_iso3_train_pred
    y_val_pred = y_iso3_val_pred
    y_test_pred = y_iso3_test_pred
    train_r2, val_r2, test_r2 = iso3_train_r2, iso3_val_r2, iso3_test_r2
    train_mae, val_mae, test_mae = iso3_train_mae, iso3_val_mae, iso3_test_mae

print(f"\nFinal Baseline Results:")
print(f"Train R²: {train_r2:.4f}, Train MAE: {train_mae:.4f}")
print(f"Val R²:   {val_r2:.4f}, Val MAE:   {val_mae:.4f}")
print(f"Test R²:  {test_r2:.4f}, Test MAE:  {test_mae:.4f}")

In [ ]:
# Plot baseline predictions vs targets
def plot_baseline_scatter(targets, predictions, title, r2_score_val):
    """Plot scatter plot for baseline predictions."""
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(targets, predictions, alpha=0.5, s=20)
    ax.plot(
        [targets.min(), targets.max()],
        [targets.min(), targets.max()],
        'r--',
        lw=2,
        label='Perfect Prediction'
    )
    ax.set_title(f'{title}\n$R^2$ = {r2_score_val:.4f}', fontsize=14)
    ax.set_xlabel('True IDP', fontsize=12)
    ax.set_ylabel('Predicted IDP', fontsize=12)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# Plot baseline results
print("\n" + "="*60)
print("BASELINE PREDICTIONS")
print("="*60)

plot_baseline_scatter(y_train, y_train_pred, 'Baseline - Train Set', train_r2)
plot_baseline_scatter(y_val, y_val_pred, 'Baseline - Validation Set', val_r2)
plot_baseline_scatter(y_test, y_test_pred, 'Baseline - Test Set', test_r2)

In [ ]:
# Step 3: Simple Neural Network with ISO3 Embedding
print("\n" + "="*60)
print("BASELINE 3: Simple NN with ISO3 Embedding")
print("="*60)

# Create simple NN architecture
class SimpleIso3NN(nn.Module):
    def __init__(self, input_dim=64, hidden_dim=128):
        super(SimpleIso3NN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    
    def forward(self, x):
        return self.net(x).squeeze()

# Prepare data as tensors
X_iso3_train_tensor = torch.tensor(X_iso3_train, dtype=torch.float32).to(device)
X_iso3_val_tensor = torch.tensor(X_iso3_val, dtype=torch.float32).to(device)
X_iso3_test_tensor = torch.tensor(X_iso3_test, dtype=torch.float32).to(device)

y_train_tensor = torch.tensor(y_train, dtype=torch.float32).to(device)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).to(device)

# Initialize model, loss, and optimizer
nn_model = SimpleIso3NN(input_dim=64, hidden_dim=128).to(device)
nn_criterion = nn.MSELoss()
nn_optimizer = torch.optim.Adam(nn_model.parameters(), lr=0.001, weight_decay=1e-5)

# Training parameters
nn_epochs = 200
patience = 20
patience_counter = 0
best_val_loss = float('inf')
nn_train_losses = []
nn_val_losses = []

print(f"\nTraining NN for {nn_epochs} epochs...")

for epoch in range(nn_epochs):
    # Training phase
    nn_model.train()
    train_output = nn_model(X_iso3_train_tensor)
    train_loss = nn_criterion(train_output, y_train_tensor)
    
    nn_optimizer.zero_grad()
    train_loss.backward()
    nn_optimizer.step()
    
    nn_train_losses.append(train_loss.item())
    
    # Validation phase
    nn_model.eval()
    with torch.no_grad():
        val_output = nn_model(X_iso3_val_tensor)
        val_loss = nn_criterion(val_output, y_val_tensor)
    
    nn_val_losses.append(val_loss.item())
    
    # Early stopping
    if val_loss.item() < best_val_loss:
        best_val_loss = val_loss.item()
        patience_counter = 0
        best_nn_state = nn_model.state_dict().copy()
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch + 1}")
            nn_model.load_state_dict(best_nn_state)
            break
    
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch + 1}/{nn_epochs}, Train Loss: {train_loss.item():.6f}, Val Loss: {val_loss.item():.6f}")

# Evaluate on all splits
nn_model.eval()
with torch.no_grad():
    y_nn_train_pred = nn_model(X_iso3_train_tensor).cpu().numpy()
    y_nn_val_pred = nn_model(X_iso3_val_tensor).cpu().numpy()
    y_nn_test_pred = nn_model(X_iso3_test_tensor).cpu().numpy()

nn_train_r2 = r2_score(y_train, y_nn_train_pred)
nn_train_mae = np.mean(np.abs(y_train - y_nn_train_pred))

nn_val_r2 = r2_score(y_val, y_nn_val_pred)
nn_val_mae = np.mean(np.abs(y_val - y_nn_val_pred))

nn_test_r2 = r2_score(y_test, y_nn_test_pred)
nn_test_mae = np.mean(np.abs(y_test - y_nn_test_pred))

print(f"\nSimple NN (ISO3) Results:")
print(f"Train R²: {nn_train_r2:.4f}, Train MAE: {nn_train_mae:.4f}")
print(f"Val R²:   {nn_val_r2:.4f}, Val MAE:   {nn_val_mae:.4f}")
print(f"Test R²:  {nn_test_r2:.4f}, Test MAE:  {nn_test_mae:.4f}")

# Comparison
print("\n" + "="*60)
print("BASELINE COMPARISON")
print("="*60)
print(f"Linear Regression (ISO3)      - Test R²: {iso3_test_r2:.4f}")
print(f"Ridge Regression (ISO3+VIIRS) - Test R²: {ridge_test_r2:.4f}")
print(f"Simple NN (ISO3)              - Test R²: {nn_test_r2:.4f}")

# Plot NN training curves
plt.figure(figsize=(10, 5))
plt.plot(nn_train_losses, label='Train Loss', linewidth=2)
plt.plot(nn_val_losses, label='Validation Loss', linewidth=2)
plt.title('Simple NN Training History', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('MSE Loss', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Plot NN predictions
plot_baseline_scatter(y_train, y_nn_train_pred, 'Simple NN Baseline - Train Set', nn_train_r2)
plot_baseline_scatter(y_val, y_nn_val_pred, 'Simple NN Baseline - Validation Set', nn_val_r2)
plot_baseline_scatter(y_test, y_nn_test_pred, 'Simple NN Baseline - Test Set', nn_test_r2)

# NN

- Architecture

In [ ]:
class MultiBranchCNN(nn.Module):
    def __init__(self, output_dim=1):
        super(MultiBranchCNN, self).__init__()

        # === ResNet-50 Feature Extractor ===
        self.rgb_model = models.resnet50(pretrained=True)
        for param in self.rgb_model.parameters():
            param.requires_grad = True
        self.rgb_model.fc = nn.Identity()
        rgb_output_dim = 2048

        # === RGB Projection Head ===
        self.rgb_proj = nn.Sequential(
            nn.Linear(rgb_output_dim, 1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),  # final output is 64
            nn.ReLU(),
        )

        # === VIIRS CNN Branch ===
        self.viirs_cnn = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),  # (32, H, W)
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # Downsample
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),  # Output: (64, 1, 1)
            nn.Flatten(),  # Final output: (64,)
        )

        # === Fusion MLP ===
        self.mlp = nn.Sequential(
            nn.Linear(64 + 64 + 64, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, output_dim),
        )

    def forward(
        self, rgb_img, viirs_img, country_embedding
    ):  # country_embedding is (batch_size, 64)
        rgb_feat = self.rgb_model(rgb_img)
        rgb_proj = self.rgb_proj(rgb_feat)  # (batch_size, 64)

        viirs_feat = self.viirs_cnn(viirs_img)  # (batch_size, 64)

        fused = torch.cat(
            [rgb_proj, viirs_feat, country_embedding], dim=1
        )  # (batch_size, 192)
        output = self.mlp(fused)
        return output.squeeze(1)  # for regression; remove this for classification


- Hyperparameters
    - last I used is 1e-5, 64 batch
    

In [ ]:
batch_size = 64  # we reached .7 training on this .15 for test (64, and 1e-5 works too)
learning_rate = 1e-3 # e-5 was fine , batch size 32 (.37 train, .09 test)
weight_decay = 1e-4 # -5 is fine. 
num_epochs = 400

- Transform

In [ ]:
target_size = (224, 224)  # Define the target size for resizing images

class JointTransform:
    def __init__(self, target_size=target_size):
        self.target_size = target_size
        self.resize_rgb = transforms.Resize(
            target_size, interpolation=transforms.InterpolationMode.BILINEAR, antialias=False
        )
        self.color_jitter = transforms.ColorJitter(
            brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1
        )
        self.normalize_rgb = transforms.Normalize(
            mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
        )
        self.normalize_viirs = transforms.Normalize(mean=[0], std=[1])

    def __call__(self, rgb, viirs):
        # Resize only RGB (PIL)
        rgb = self.resize_rgb(rgb)
        rgb = self.color_jitter(rgb)
        
        # Random horizontal flip
        if random.random() > 0.5:
            rgb = TF.hflip(rgb)
            viirs = TF.hflip(viirs)

        # Random vertical flip
        if random.random() > 0.5:
            rgb = TF.vflip(rgb)
            viirs = TF.vflip(viirs)

        # Normalize
        rgb = self.normalize_rgb(rgb)
        # viirs = self.normalize_viirs(viirs)

        return rgb, viirs


- Dataset and DataLoader

In [ ]:
class IDPDataset(Dataset):
    def __init__(self, df, joint_transform=None):
        self.df = df
        self.joint_transform = joint_transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        viirs = (
            torch.tensor(self.df.iloc[idx]["viirs_diff"]).float().squeeze().to(device)
        )

        if viirs.ndim == 2:
            viirs = viirs.unsqueeze(0)  # (1, H, W)

        rgb = torch.tensor(self.df.iloc[idx]["rgb"]).float().squeeze().to(device)

        target = torch.tensor(self.df.iloc[idx]["figures"]).float().to(device)
        iso3 = self.df.iloc[idx]["iso3"]
        iso3_emb = torch.tensor(self.df.iloc[idx]["iso3_encoded"]).to(device)

        if self.joint_transform:
            rgb, viirs = self.joint_transform(rgb, viirs)

        return rgb, viirs, target, iso3, iso3_emb

In [ ]:
# Create the transform
joint_transform = JointTransform(target_size=(224, 224))

# Create the dataset
dataset = IDPDataset(df=df, joint_transform=joint_transform)

In [ ]:
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, generator=generator)

- Data split

In [ ]:
def create_data_loaders(dataset, batch_size, generator, split_ratio=(0.8, 0.1, 0.1)):
    assert sum(split_ratio) == 1.0, "Split ratios must sum to 1.0"

    total = len(dataset)
    train_size = int(split_ratio[0] * total)
    val_size = int(split_ratio[1] * total)
    test_size = total - train_size - val_size  # Ensure full coverage

    train_ds, val_ds, test_ds = torch.utils.data.random_split(
        dataset, [train_size, val_size, test_size], generator=generator
)

    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True, generator=generator),
        DataLoader(val_ds, batch_size=batch_size, shuffle=False),
        DataLoader(test_ds, batch_size=batch_size, shuffle=False),
)

In [ ]:
train_loader, val_loader, test_loader = create_data_loaders(
    dataset, batch_size, generator
)

- Initiate model, loss and optimizer

In [ ]:
model = MultiBranchCNN(output_dim=1).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

# scheduler = ReduceLROnPlateau(
#     optimizer, mode="min", factor=0.5, patience=5, verbose=True
# )

In [ ]:
summary(
    model,
    input_size=(
        ( batch_size, 3, 224, 224),
        ( batch_size, 1, 20, 20),
        ( batch_size, 64)
    ),
    device=device.type,
)

# Training

In [ ]:
train_size = len(train_loader.dataset)
val_size = len(val_loader.dataset)

In [ ]:
best_loss = float("inf")
patience = 10
patience_counter = 0
train_losses = []
val_losses = []

print(f"Training on {train_size} examples, validating on {val_size} examples")

for epoch in range(num_epochs):

    for param_group in optimizer.param_groups:
        print(f"Current learning rate: {param_group['lr']}")
        
    # Training phase
    model.train()
    epoch_train_loss = 0.0

    for rgb, viirs, targets, _ , iso3_emb in train_loader:
        # Move data to device
        rgb = rgb.to(device)
        viirs = viirs.to(device)
        targets = targets.to(device)
        iso3_emb = iso3_emb.to(device)

        # Forward pass
        outputs = model(rgb, viirs, iso3_emb)
        loss = criterion(outputs, targets)
        epoch_train_loss += loss.item()

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    epoch_train_loss /= len(train_loader)
    train_losses.append(epoch_train_loss)

    # Validation phase
    model.eval()
    epoch_val_loss = 0.0

    with torch.no_grad():
        for rgb, viirs, targets, _, iso3_emb in val_loader:
            rgb = rgb.to(device)
            viirs = viirs.to(device)
            targets = targets.to(device)
            iso3_emb = iso3_emb.to(device)

            outputs = model(rgb, viirs, iso3_emb)
            val_loss = criterion(outputs, targets)
            epoch_val_loss += val_loss.item()

    epoch_val_loss /= len(val_loader)
    val_losses.append(epoch_val_loss)
    # scheduler.step(epoch_val_loss)

    # Print epoch statistics
    print(
        f"Epoch [{epoch + 1}/{num_epochs}], Train Loss: {epoch_train_loss:.4f}, Val Loss: {epoch_val_loss:.4f}"
    )

    # Early stopping
    if epoch_val_loss < best_loss:
        best_loss = epoch_val_loss
        patience_counter = 0
        # Save best model
        torch.save(model.state_dict(), folder_path + "checkpoints/best_model_gidd.pth")
        print(f" ✅ Saved with validation loss: {best_loss:.4f}")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping triggered after {epoch + 1} epochs")
            break

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label="Training Loss")
plt.plot(val_losses, label="Validation Loss")
plt.title("Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# if using ReduceLROnPlateau, restore lr
if "scheduler" in locals():
    for param_group in optimizer.param_groups:
        param_group["lr"] = learning_rate  # Reset learning rate to initial value

# Load best model

- Inference using best model

In [ ]:
model.load_state_dict(torch.load(folder_path + "checkpoints/best_model_gidd.pth"))

In [ ]:
def predict_batch(model, batch, device):
    """Processes a single batch and returns predictions and targets."""
    rgb, viirs, target, _, iso3_emb = batch
    rgb = rgb.to(device)
    viirs = viirs.to(device)
    iso3_emb = iso3_emb.to(device)

    output = model(rgb, viirs, iso3_emb)
    return output.cpu().numpy(), target.cpu().numpy()


def get_predictions_and_targets(model, dataloader, device):
    """Gets predictions and targets for the entire dataloader."""
    model.eval()
    predictions, targets = [], []

    with torch.no_grad():
        for batch in dataloader:
            batch_preds, batch_targets = predict_batch(model, batch, device)
            predictions.extend(batch_preds)
            targets.extend(batch_targets)

    return np.array(predictions), np.array(targets)


def plot_scatter(ax, targets, predictions, title):
    """Plots a scatter plot with predictions vs targets."""
    r2 = r2_score(targets, predictions)
    ax.scatter(targets, predictions, alpha=0.3)
    ax.plot(
        [targets.min(), targets.max()],
        [targets.min(), targets.max()],
        "r--",
        lw=2,
    )
    ax.set_title(f"{title}\n$R^2$ = {r2:.4f}")
    ax.set_xlabel("True IDP")
    ax.set_ylabel("Predicted IDP")


def plot_predictions_vs_targets(
    model, device, *data_loaders, save=False, save_path="predictions_vs_targets.pdf"
):

    num_sets = len(data_loaders)
    fig, axes = plt.subplots(
        1, num_sets, figsize=(7 * num_sets, 6), sharex=True, sharey=True
    )

    if num_sets == 1:
        axes = [axes] 

    for ax, (label, loader) in zip(axes, data_loaders):
        preds, targets = get_predictions_and_targets(model, loader, device)
        plot_scatter(ax, targets, preds, f"{label} Set")

    # plt.suptitle("Predictions vs Targets")
    plt.tight_layout(rect=[0, 0, 1, 0.95])

    if save:
        os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)
        plt.savefig(save_path, format=save_path.split(".")[-1], bbox_inches="tight")
        print(f"Plot saved to: {os.path.abspath(save_path)}")
        plt.show(block=False)  # Show the plot without blocking
        plt.close()
    else:
        plt.show()

In [ ]:
plot_predictions_vs_targets(
    model, device, ("Train", train_loader), ("Test", test_loader), save=True, save_path="predictions_vs_targets.pdf"
)

In [ ]:
plot_predictions_vs_targets(model, device, ("Test", test_loader), save=True, save_path="scatter_idmc.pdf")

# IDU

In [ ]:
path_idu = "../src/data/processed/testing.h5"
df_idu = read_hdf5_to_dataframe_with_index(path_idu)

In [ ]:
df_idu, le_idu, emb_idu = replace_iso3_with_embedding(df_idu)
df_idu = create_viirs_diff_column(df_idu)
df_idu = log_and_normalize_column(df_idu, "figures")

In [ ]:
dataset_idu = IDPDataset(df=df_idu, joint_transform=joint_transform)

In [ ]:
dataloader_idu = DataLoader(dataset_idu, batch_size=batch_size, shuffle=True, generator=generator)

In [ ]:
# split the dataset into train, val, test
train_loader_idu, val_loader_idu, test_loader_idu = create_data_loaders(dataset_idu, batch_size, generator)

In [ ]:
train_size_idu = len(train_loader_idu.dataset)
val_size_idu = len(val_loader_idu.dataset)
test_size_idu = len(test_loader_idu.dataset)

- Fine-tuning

In [ ]:
# finetune model on idu
best_loss = float("inf")
patience = 10
patience_counter = 0
train_losses = []
val_losses = []

print(f"Training on {train_size_idu} examples, validating on {val_size_idu} examples")

for epoch in range(num_epochs):
    for param_group in optimizer.param_groups:
        print(f"Current learning rate: {param_group['lr']}")

    # Training phase
    model.train()
    epoch_train_loss = 0.0

    for rgb, viirs, targets, _, iso3_emb in train_loader_idu:
        # Move data to device
        rgb = rgb.to(device)
        viirs = viirs.to(device)
        targets = targets.to(device)
        iso3_emb = iso3_emb.to(device)

        # Forward pass
        outputs = model(rgb, viirs, iso3_emb)
        loss = criterion(outputs, targets)
        epoch_train_loss += loss.item()

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    epoch_train_loss /= len(train_loader)
    train_losses.append(epoch_train_loss)

    # Validation phase
    model.eval()
    epoch_val_loss = 0.0

    with torch.no_grad():
        for rgb, viirs, targets, _, iso3_emb in val_loader_idu:
            rgb = rgb.to(device)
            viirs = viirs.to(device)
            targets = targets.to(device)
            iso3_emb = iso3_emb.to(device)

            outputs = model(rgb, viirs, iso3_emb)
            val_loss = criterion(outputs, targets)
            epoch_val_loss += val_loss.item()

    epoch_val_loss /= len(val_loader)
    val_losses.append(epoch_val_loss)
    # scheduler.step(epoch_val_loss)

    # Print epoch statistics
    print(
        f"Epoch [{epoch + 1}/{num_epochs}], Train Loss: {epoch_train_loss:.4f}, Val Loss: {epoch_val_loss:.4f}"
    )

    # Early stopping
    if epoch_val_loss < best_loss:
        best_loss = epoch_val_loss
        patience_counter = 0
        # Save best model
        torch.save(model.state_dict(), folder_path + "checkpoints/best_model_idu.pth")
        print(f" ✅ Saved with validation loss: {best_loss:.4f}")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping triggered after {epoch + 1} epochs")
            break

In [ ]:
plot_predictions_vs_targets(model, device, ("Test", test_loader_idu), save=True, save_path="scatter_idu.pdf")

In [ ]:
# combine test_loader and test_loader_idu into one dataloader and plot predictions vs targets for both
combined_dataset = torch.utils.data.ConcatDataset([test_loader.dataset, test_loader_idu.dataset])
combined_test_loader = DataLoader(combined_dataset, batch_size=batch_size, shuffle=False)

- Per country error

In [ ]:
def evaluate_per_country(model, dataloader):
    model.eval()
    country_predictions = {}
    country_targets = {}

    with torch.no_grad():
        for rgb, viirs, target, iso, iso3_emb in dataloader:
            rgb = rgb.to(device)
            viirs = viirs.to(device)
            iso3_emb = iso3_emb.to(device)  # keep on same device as model
            output = model(rgb, viirs, iso3_emb)

            output_cpu = output.cpu()
            target_cpu = target.cpu()

            for i in range(len(iso)):
                code = iso[i]
                if code not in country_predictions:
                    country_predictions[code] = []
                    country_targets[code] = []
                country_predictions[code].append(output_cpu[i].item())
                country_targets[code].append(target_cpu[i].item())

    return country_predictions, country_targets


country_predictions, country_targets = evaluate_per_country(model, combined_test_loader)

country_mae = {
    iso3: np.mean(
        np.abs(
            np.array(country_predictions[iso3], dtype=np.float32)
            - np.array(country_targets[iso3], dtype=np.float32)
        )
    )
    for iso3 in country_predictions
}

mae_df = pd.DataFrame({"iso3": list(country_mae.keys()), "mae": list(country_mae.values())})
mae_df = mae_df[mae_df["iso3"] != "ATA"]  # optional

fig = px.choropleth(
    mae_df,
    locations="iso3",
    color="mae",
    hover_name="iso3",
    color_continuous_scale="Reds",
)

fig.update_geos(
    showframe=False,
    showcoastlines=False,
    projection_type="equirectangular",
    lataxis_range=[-58, 90],  # hides Antarctica
)
fig.update_layout(margin={"r": 0, "t": 50, "l": 0, "b": 0})

fig.show()

# save as pdf
fig.write_image("mae_per_country.pdf", format="pdf", scale=2)


In [ ]:
# which countries have the highest MAE
highest_mae_countries = mae_df.sort_values(by="mae", ascending=False).head(20)
print("Countries with the highest MAE:")
print(highest_mae_countries)

In [ ]:
# see overlapping countries that are highest MAE and least represented in df
overlapping_countries = set(highest_mae_countries["iso3"]).intersection(
    set(list_least_frequent_iso3(df, threshold=20)))

In [ ]:
list_least_frequent_iso3(df, threshold=20)